# E-commerce Consumer Behavior Analysis

### Load and inspect the raw data

We will first check the file's size, columns, data types, and missing values. These checks help us decide what to clean rather than changing data blindly.

In [2]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
PROJECT_ROOT = Path.cwd()
RAW_DATA_FILE = PROJECT_ROOT / "data" / "Ecommerce_Consumer_Behavior_Analysis_Data.csv"
CLEAN_DATA_FILE = PROJECT_ROOT/"data"/"Ecommerce_Consumer_Behavior_Analysis_Data_clean.csv"

sns.set_theme(style='whitegrid', palette='deep')
print(f"Raw file: {RAW_DATA_FILE}")
print(f"File exist: {RAW_DATA_FILE.exists()}")

Raw file: e:\python\archive\data\Ecommerce_Consumer_Behavior_Analysis_Data.csv
File exist: True


In [8]:
raw_df = pd.read_csv(RAW_DATA_FILE)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
display(raw_df.head())

Rows: 1,000
Columns: 28


,Customer_ID,Age,Gender,Income_Level,Marital_Status,Education_Level,Occupation,Location,Purchase_Category,Purchase_Amount,...,Customer_Satisfaction,Engagement_with_Ads,Device_Used_for_Shopping,Payment_Method,Time_of_Purchase,Discount_Used,Customer_Loyalty_Program_Member,Purchase_Intent,Shipping_Preference,Time_to_Decision
0,37-611-6911,22,Female,Middle,Married,Bachelor's,Middle,Évry,Gardening & Outdoors,$333.80,...,7,NaN,Tablet,Credit Card,3/1/2024,True,False,Need-based,No Preference,2
1,29-392-9296,49,Male,High,Married,High School,High,Huocheng,Food & Beverages,$222.22,...,5,High,Tablet,PayPal,4/16/2024,True,False,Wants-based,Standard,6
2,84-649-5117,24,Female,Middle,Single,Master's,High,Huzhen,Office Supplies,$426.22,...,7,Low,Smartphone,Debit Card,3/15/2024,True,True,Impulsive,No Preference,3
3,48-980-6078,29,Female,Middle,Single,Master's,Middle,Wiwilí,Home Appliances,$101.31,...,1,NaN,Smartphone,Other,10/4/2024,True,True,Need-based,Express,10
4,91-170-9072,33,Female,Middle,Widowed,High School,Middle,Nara,Furniture,$211.70,...,10,NaN,Smartphone,Debit Card,1/30/2024,False,False,Wants-based,No Preference,4


In [9]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 28 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Customer_ID                            1000 non-null   str    
 1   Age                                    1000 non-null   int64  
 2   Gender                                 1000 non-null   str    
 3   Income_Level                           1000 non-null   str    
 4   Marital_Status                         1000 non-null   str    
 5   Education_Level                        1000 non-null   str    
 6   Occupation                             1000 non-null   str    
 7   Location                               1000 non-null   str    
 8   Purchase_Category                      1000 non-null   str    
 9   Purchase_Amount                        1000 non-null   str    
 10  Frequency_of_Purchase                  1000 non-null   int64  
 11  Purchase_Channel

In [21]:
quality_before = pd.DataFrame({
    "missing_values": raw_df.isna().sum(), 
    "missing_percent": (raw_df.isna().mean()*100).round(1)    
})

print(f"Exact Duplicate Data: {raw_df.duplicated().sum():,}")
print(f"Duplicate customer ID: {raw_df["Customer_ID"].duplicated().sum():,}")
display(quality_before[quality_before['missing_values'] > 0])

Exact Duplicate Data: 0
Duplicate customer ID: 0


,missing_values,missing_percent
Social_Media_Influence,247,24.7
Engagement_with_Ads,256,25.6


### Cleaning

The purchase amount includes currency symbols and extra spaces, and the purchase date is text. We will convert those columns to useful types. Missing social-media and ad-engagement answers are kept as **`Unknown`** so they are not confused with a low score or silently dropped. We will preserve unusual category labels (including `Packages)`) because we cannot safely guess their intended spelling.

In [22]:
clean_df = raw_df.copy()
clean_df.head(3)

,Customer_ID,Age,Gender,Income_Level,Marital_Status,Education_Level,Occupation,Location,Purchase_Category,Purchase_Amount,...,Customer_Satisfaction,Engagement_with_Ads,Device_Used_for_Shopping,Payment_Method,Time_of_Purchase,Discount_Used,Customer_Loyalty_Program_Member,Purchase_Intent,Shipping_Preference,Time_to_Decision
0,37-611-6911,22,Female,Middle,Married,Bachelor's,Middle,Évry,Gardening & Outdoors,$333.80,...,7,NaN,Tablet,Credit Card,3/1/2024,True,False,Need-based,No Preference,2
1,29-392-9296,49,Male,High,Married,High School,High,Huocheng,Food & Beverages,$222.22,...,5,High,Tablet,PayPal,4/16/2024,True,False,Wants-based,Standard,6
2,84-649-5117,24,Female,Middle,Single,Master's,High,Huzhen,Office Supplies,$426.22,...,7,Low,Smartphone,Debit Card,3/15/2024,True,True,Impulsive,No Preference,3


In [34]:
# Remove accidental spaces around text values, without changing there meaning.
text_column = [
    column for column in clean_df.columns
    if pd.api.types.is_string_dtype(clean_df[column].dtype)
]

for column in text_column:
    clean_df[column] = clean_df[column].str.strip()

In [47]:
# the source use "None" for non answered survey field; 

survey_columns = ['Social_Media_Influence', 'Engagement_with_Ads']
for column in survey_column:
    clean_df[column] = clean_df[column].fillna("Unknown")

In [48]:
# correcting the Purchase_Amount

clean_df['Purchase_Amount'] = pd.to_numeric(clean_df['Purchase_Amount'].astype('string').str.replace(r"[$,\s]", "", regex=True), errors="coerce",)

clean_df["Purchase_Amount"].head(3)

0     333.8
1    222.22
2    426.22
Name: Purchase_Amount, dtype: Float64

In [49]:
# purchase time string to datetime

clean_df["Time_of_Purchase"] = pd.to_datetime(clean_df["Time_of_Purchase"], format="mixed", errors="coerce")

clean_df["Time_of_Purchase"].head(3)

0   2024-03-01
1   2024-04-16
2   2024-03-15
Name: Time_of_Purchase, dtype: datetime64[us]

In [50]:
rows_before_duplicate = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
duplicates_removed = rows_before_duplicate - len(clean_df)

In [51]:
print(f"Exact  duplicate rows removed: {duplicates_removed}:,")
print(f"cleaned rows: {len(clean_df):,}")
print(f"Unparsed purchase amounts: {clean_df['Purchase_Amount'].isna().sum():,}")
print(f"Unparsed purchase dates: {clean_df['Time_of_Purchase'].isna().sum():,}")
display(clean_df[["Purchase_Amount", "Time_of_Purchase", *survey_columns]].head())

Exact  duplicate rows removed: 0:,
cleaned rows: 1,000
Unparsed purchase amounts: 0
Unparsed purchase dates: 0


,Purchase_Amount,Time_of_Purchase,Social_Media_Influence,Engagement_with_Ads
0,333.8,2024-03-01,Unknown,Unknown
1,222.22,2024-04-16,Medium,High
2,426.22,2024-03-15,Low,Low
3,101.31,2024-10-04,High,Unknown
4,211.7,2024-01-30,Medium,Unknown


In [52]:
# saving the cleaned file

CLEAN_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(CLEAN_DATA_FILE, index=False, date_format="%Y-%m-%d", encoding="utf-8-sig")

print(f"Saved cleaned file: {CLEAN_DATA_FILE}")
print(f"cleaned file size: {CLEAN_DATA_FILE.stat().st_size:,} bytes")

Saved cleaned file: e:\python\archive\data\Ecommerce_Consumer_Behavior_Analysis_Data_clean.csv
cleaned file size: 196,537 bytes
